In [ ]:
# Module 10 — Code Along: the LLM as a task engine
# Real OpenAI calls — instructor runs live, students follow along.
import json, os
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

client = OpenAI()   # reads OPENAI_API_KEY from env

TICKETS = [
    {"id": 1, "subject": "Can't log in after password reset",  "category": "Account",   "priority": "high"},
    {"id": 2, "subject": "Invoice double-charged this month",  "category": "Billing",   "priority": "urgent"},
    {"id": 3, "subject": "How do I export my data?",           "category": "Technical", "priority": "low"},
    {"id": 4, "subject": "App crashes on upload",              "category": "Technical", "priority": "high"},
    {"id": 5, "subject": "Refund not received",                "category": "Billing",   "priority": "normal"},
]

def ask(system: str, user: str, *, json_mode: bool = False) -> str:
    """One LLM call — the shape you'll use all day."""
    kwargs = {}
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        **kwargs,
    )
    return resp.choices[0].message.content

print(f"OpenAI client ready, {len(TICKETS)} sample tickets loaded.")

# Section 1 · Closed-source API power

One hosted engine, many tasks. Same `chat.completions.create` call — the **task lives in the prompt**, not the API. Each cell below is a different task on the same ticket data.

In [ ]:
# TASK = CLASSIFY — subject in, structured label out.
ticket = TICKETS[1]
label = ask(
    "Classify this support ticket's priority: low, normal, high, or urgent. Reply with just the label.",
    ticket["subject"],
)
print(f"Subject: {ticket['subject']}")
print(f"Model:   {label}")
print(f"Actual:  {ticket['priority']}")

In [ ]:
# Classify ALL tickets — same one-line call, different input each time.
for t in TICKETS:
    pred = ask("Classify priority: low/normal/high/urgent. Reply with just the label.",
              t["subject"])
    match = "✓" if pred.strip().lower() == t["priority"] else "✗"
    print(f"{match} #{t['id']} predicted={pred.strip():8s} actual={t['priority']}")

## TASK = SUMMARIZE — collapse a thread to its gist

A support thread can be dozens of turns. Summarize compresses it to one actionable line. Same API shape: text in, shorter text out.

In [ ]:
# TASK = SUMMARIZE — multi-turn thread in, one line out.
thread = [
    "Customer: I was charged twice for my June invoice.",
    "Agent: I see two charges of $49 on the 3rd. Let me check.",
    "Customer: Yes, please refund the duplicate.",
    "Agent: Refund issued, 3-5 business days.",
]

summary = ask("Summarize this support thread in one line.",
              "\n".join(thread))
print("SUMMARY:", summary)

### Same shape, different vendor (Gemini)

Google's Gemini API has the same request shape — a model id, content in, text out. We stay on OpenAI for M11/M12, but the *pattern* is portable.

```python
from google import genai
client = genai.Client()                          # GOOGLE_API_KEY in env
resp = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Classify this ticket: my screen is cracked",
)
resp.text                                        # -> "hardware"
```

# Section 2 · Open-source LLM power

Models you run yourself — no API key, data stays local. `transformers.pipeline()` wraps load → preprocess → infer → postprocess into one call.

Set `RUN_LIVE_HF=1` to actually download and run. The models are small enough for CPU.

In [ ]:
# LIVE: zero-shot classification — no training data, you supply the labels.
RUN_LIVE_HF = os.getenv("RUN_LIVE_HF") == "1"

if RUN_LIVE_HF:
    from transformers import pipeline
    clf = pipeline("zero-shot-classification",
                   model="typeform/distilbert-base-uncased-mnli")
    out = clf("My invoice is wrong and I was double charged",
              candidate_labels=["billing", "hardware", "account", "urgent"])
    print(list(zip(out["labels"], [round(s, 3) for s in out["scores"]])))
else:
    print("RUN_LIVE_HF!=1 — skipping download. Expected output:")
    print("[('billing', 0.94), ('account', 0.04), ('urgent', 0.01), ('hardware', 0.01)]")

### Shown, not run — heavier modalities

Same `pipeline()` call; the models are multi-GB so we show expected outputs.

```python
# Speech → text
pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")("meeting.flac")
# >>> {'text': 'GOING ALONG SLUSHY COUNTRY ROADS AND SPEAKING TO DAMP AUDIENCES ...'}

# Image → caption
pipeline("image-to-text", model="Salesforce/blip-image-captioning-base")("parrots.png")
# >>> [{'generated_text': 'two birds are standing next to each other'}]

# Translation EN→ES
pipeline("translation", model="Helsinki-NLP/opus-mt-en-es")("Hello, how are you today?")
# >>> [{'translation_text': 'Hola, ¿cómo estás hoy?'}]
```

# Section 3 · Your first LLM app

The part Lab 10 makes you build. Take a raw call and make it **structured + validated**. The LLM is just another untrusted source — validate at the boundary, exactly like a request body.

## Structured output — make the LLM speak your schema

A free-text answer is hard to use programmatically. A **Pydantic model** turns the LLM into a typed function:

- `Field(description=...)` tells the model what each field means
- `response_format={"type": "json_object"}` guarantees valid JSON from the API
- `model_validate_json(raw)` catches anything the model got wrong

The schema is both the *instruction* to the model **and** the *validator* on the way back.

In [ ]:
# The extraction target — every field's description doubles as instruction to the LLM.
class TicketQuery(BaseModel):
    category: str | None = Field(None, description="Restrict to this category, or null for all.")
    priority: str | None = Field(None, description="One of low|normal|high|urgent, or null.")
    open_only: bool = False

# model_json_schema() is what you'd send as the JSON-mode schema.
print(json.dumps(TicketQuery.model_json_schema(), indent=2))

In [ ]:
# TASK = EXTRACT + VALIDATE — natural language in, validated TicketQuery out.
schema_str = json.dumps(TicketQuery.model_json_schema())

raw = ask(
    f"Extract a TicketQuery from the user's question. Reply as JSON matching this schema:\n{schema_str}",
    "show me open billing tickets",
    json_mode=True,
)

print("Raw JSON:", raw)
query = TicketQuery.model_validate_json(raw)    # validate at the boundary
print("Parsed:  ", query)
print("category is a real str:", type(query.category).__name__)

In [ ]:
# Try a few more natural-language queries:
questions = [
    "urgent technical issues",
    "everything that's still open",
    "low priority billing",
]

for q in questions:
    raw = ask(
        f"Extract a TicketQuery as JSON matching this schema:\n{schema_str}",
        q,
        json_mode=True,
    )
    parsed = TicketQuery.model_validate_json(raw)
    print(f"{q:40s} → {parsed}")

## Always validate — the LLM is just another untrusted source

JSON mode guarantees valid JSON — **not** valid *schema*. The LLM can still return extra fields, wrong types, or values outside your constraints. `model_validate_json()` catches all of that, exactly like a Day-2 request body.

Data crossing into your system is guilty until validated. The LLM is no different.

In [ ]:
# What happens when the LLM returns bad JSON? Same boundary discipline.
bad_responses = [
    '{"category": "Billing", "open_only": "yes"}',    # wrong type: str instead of bool
    '{"category": "Billing", "open_only":',             # truncated JSON
    '{"nonsense_field": 42}',                            # unknown field (passes — extras ignored)
]

for bad in bad_responses:
    try:
        q = TicketQuery.model_validate_json(bad)
        print(f"OK:   {bad:55s} → {q}")
    except (ValidationError, Exception) as e:
        print(f"CAUGHT: {bad:53s} → {type(e).__name__}")

## The improvement ladder (concept — shown, not run)

Output not good enough? Climb only as far as the task needs:

| Rung | Fix it by… | When |
|---|---|---|
| **Prompt** | clearer role / constraints / examples | first, always |
| **RAG** | retrieve facts, inject as context | model lacks *your* knowledge |
| **Fine-tune** | retrain weights on labeled data | one narrow task, high volume, fixed format |

Most tasks never leave rung 1. The held-out check that fine-tuning needs is exactly the **golden-eval you build in M12**.

## Recap — Module 10

- **§1 Closed API** — one hosted engine, many tasks (classify / summarize); same shape across vendors
- **§2 Open-source** — `pipeline()` runs a task locally, no key; pick closed for reasoning, open for a fixed high-volume task
- **§3 Your first app** — structured output + **validate at the boundary**; the LLM is untrusted, just like any external source

→ **Lab 10**: Part A classify a Product (open-source), Part B build `parse_nl_query` + `apply_query`.
→ **M11**: wrap this one call in a loop with tools.